# TF-IDF + Linear SVM — Many-to-Many (MTM, cyclic)

Trains a One-vs-Rest Logistic Regression classifier (class-weight balanced) on
TF-IDF features over a cyclic per-document language rotation across all 23 languages of MultiEURLEX, then evaluates on all 23
languages of the test set.

In [ ]:
!pip -q install -U datasets scikit-learn

from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/multieurlex_all_languages")

label_feature = dataset["train"].features["labels"].feature
print("Train / Val / Test sizes:",
      dataset["train"].num_rows, dataset["validation"].num_rows, dataset["test"].num_rows)
print("Total possible label classes:", len(label_feature.names))

## Configuration

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

SEED = 42
TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"

## Label vocabulary

In [ ]:
all_label_ids = sorted({
    int(label_id)
    for row in dataset["train"]["labels"]
    for label_id in row
})
NUM_LABELS = len(all_label_ids)
label_to_index = {label_id: i for i, label_id in enumerate(all_label_ids)}

print("Number of labels:", NUM_LABELS)

def multi_hot(label_ids, mapping):
    vector = np.zeros(len(mapping), dtype=np.float32)
    for label_id in label_ids:
        vector[mapping[int(label_id)]] = 1.0
    return vector

## Text + label extraction (cyclic language rotation)

In [ ]:
LANGUAGES = ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'hr',
             'hu', 'it', 'lt', 'lv', 'mt', 'nl', 'pl', 'pt', 'ro', 'sk', 'sl', 'sv']

def assign_language(idx):
    # Deterministic: document i always gets the same language. There is no
    # epoch dimension here (the model is fit once, not over multiple epochs),
    # so a single cyclic pass across the training set is used.
    return LANGUAGES[idx % len(LANGUAGES)]

def prepare_xy(df):
    X = []
    for i, item in enumerate(df[TEXT_COLUMN]):
        lang = assign_language(i)
        text = item.get(lang, "") or item.get("en", "")  # fallback to English if missing
        X.append(text)
    Y = np.vstack([
        multi_hot([lbl for lbl in row if int(lbl) in label_to_index], label_to_index)
        for row in df[LABEL_COLUMN]
    ])
    return X, Y

# Processed in two halves to limit peak RAM
train_split_size = len(dataset["train"])
half = train_split_size // 2
train_df_part1 = dataset["train"].select(range(half)).to_pandas()
train_df_part2 = dataset["train"].select(range(half, train_split_size)).to_pandas()

X_train_part1, y_train_part1 = prepare_xy(train_df_part1)
# NOTE: each half's cyclic index restarts at 0 (prepare_xy uses the dataframe's
# own local row position), but since both halves are equal-sized and cycle
# through the same 23 languages, the overall rotation is still balanced.
X_train_part2, y_train_part2 = prepare_xy(train_df_part2)

X_train = X_train_part1 + X_train_part2
y_train = np.vstack([y_train_part1, y_train_part2])
print("X_train length:", len(X_train), "| y_train shape:", y_train.shape)

## TF-IDF vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

# Fit/transform in four chunks to limit peak RAM
quarter = len(X_train) // 4
X_train_parts = [
    X_train[:quarter], X_train[quarter:2*quarter],
    X_train[2*quarter:3*quarter], X_train[3*quarter:],
]

from scipy.sparse import vstack
X_train_tfidf_parts = []
for i, part in enumerate(X_train_parts):
    X_train_tfidf_parts.append(tfidf.fit_transform(part) if i == 0 else tfidf.transform(part))
X_train_tfidf = vstack(X_train_tfidf_parts)

print("TF-IDF train shape:", X_train_tfidf.shape)

## Train — One-vs-Rest Logistic Regression

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

baseline_model = OneVsRestClassifier(
    LinearSVC(max_iter=1000, class_weight="balanced"),
    n_jobs=-1,
)

start = time.perf_counter()
baseline_model.fit(X_train_tfidf, y_train)
baseline_train_time = time.perf_counter() - start

print(f"Training time: {baseline_train_time:.2f} seconds")

## Metrics

In [ ]:
def multilabel_metrics(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred),
    }

## Multilingual evaluation (all 23 languages)

In [ ]:
from tqdm.auto import tqdm

LANGUAGES = ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'hr',
             'hu', 'it', 'lt', 'lv', 'mt', 'nl', 'pl', 'pt', 'ro', 'sk', 'sl', 'sv']

def evaluate_baseline_per_language(model, vectorizer, raw_dataset, label_to_index, languages):
    results = {}
    for lang in tqdm(languages, desc="Languages (TF-IDF + SVM eval)"):
        texts, label_rows = [], []
        for row in raw_dataset:
            text = row[TEXT_COLUMN].get(lang)
            if text:
                texts.append(text)
                mh = multi_hot(
                    [lbl for lbl in row[LABEL_COLUMN] if int(lbl) in label_to_index],
                    label_to_index
                )
                label_rows.append(mh)
        if not texts:
            continue

        X_lang_tfidf = vectorizer.transform(texts)
        preds = model.predict(X_lang_tfidf)
        y_true = np.vstack(label_rows)
        results[lang] = (preds, y_true)
    return results

baseline_lang_results = evaluate_baseline_per_language(
    baseline_model, tfidf, dataset["test"], label_to_index, LANGUAGES
)

## Per-language, pooled and averaged metrics

In [ ]:
per_lang_rows = []
for lang, (preds, y_true) in baseline_lang_results.items():
    m = multilabel_metrics(y_true, preds)
    per_lang_rows.append({
        "language": lang,
        "n_examples": len(y_true),
        "micro_precision": m["micro_precision"],
        "micro_recall": m["micro_recall"],
        "micro_f1": m["micro_f1"],
        "macro_precision": m["macro_precision"],
        "macro_recall": m["macro_recall"],
        "macro_f1": m["macro_f1"],
    })

baseline_per_lang_df = pd.DataFrame(per_lang_rows)
print("=== TF-IDF + SVM: per-language metrics ===")
display(baseline_per_lang_df.round(4))

pooled_preds = np.vstack([v[0] for v in baseline_lang_results.values()])
pooled_true = np.vstack([v[1] for v in baseline_lang_results.values()])

baseline_pooled_metrics = multilabel_metrics(pooled_true, pooled_preds)
print("\n=== TF-IDF + SVM: POOLED metrics across all 23 languages ===")
display(pd.Series(baseline_pooled_metrics).round(4))

baseline_avg_metrics = baseline_per_lang_df[[
    "micro_precision", "micro_recall", "micro_f1",
    "macro_precision", "macro_recall", "macro_f1"
]].mean()
print("\n=== TF-IDF + SVM: AVERAGED across the 23 per-language scores ===")
display(baseline_avg_metrics.round(4))

baseline_per_lang_df.to_csv("/content/drive/MyDrive/langF1(TFIDF-SVM)M-T-M.csv", index=False)
print("Saved per-language metrics to Drive.")

## Training time, inference time, parameter count

In [ ]:
# --- Inference time: predict on the full pooled 23-language test matrix ---
X_pooled_tfidf_list = []
for lang in LANGUAGES:
    texts = [row[TEXT_COLUMN].get(lang) for row in dataset["test"] if row[TEXT_COLUMN].get(lang)]
    if texts:
        X_pooled_tfidf_list.append(tfidf.transform(texts))
from scipy.sparse import vstack as sparse_vstack
X_pooled_tfidf = sparse_vstack(X_pooled_tfidf_list)

start = time.perf_counter()
_ = baseline_model.predict(X_pooled_tfidf)
baseline_inference_time = time.perf_counter() - start

print(f"Training time: {baseline_train_time:.2f} seconds")
print(f"Inference time (all 23 languages, {X_pooled_tfidf.shape[0]} documents): {baseline_inference_time:.2f} seconds")
print(f"Documents/sec: {X_pooled_tfidf.shape[0] / baseline_inference_time:.2f}")

# Each of the NUM_LABELS binary classifiers has one coefficient per TF-IDF feature plus one intercept.
total_params = sum(est.coef_.size + est.intercept_.size for est in baseline_model.estimators_)
print(f"\nTotal learned parameters (all {len(baseline_model.estimators_)} binary classifiers): {total_params:,}")
print(f"TF-IDF vocabulary size (feature count): {len(tfidf.vocabulary_):,}")